# Klasik Görüntü İşleme ile Mikroskobik Görüntülerde Hücre Sayımı ve Zamansal Analiz
**Veri seti:** Cell Tracking Challenge — Fluo-N2DH-GOWT1 (eğitim arşivi, sekans 01 ve 02)
**Öğrenci no:** 244329059 · **Seed:** 29059 + 2026 = **31085**

Notebook iki bölümden oluşur:

* **BÖLÜM A (0–11):** veri, seçim, referanslar (yalnızca geliştirme + doğrulama), baseline, kontrollü deneyler,
  doğrulama, nihai pipeline'ın kilitlenmesi ve test tahminlerinin **referanslar okunmadan** kaydedilmesi.
  → Bölüm A bitince `final_pipeline.json`, `test_predictions_locked.csv`, `test_lock.sha256` GitHub'a **ayrı bir commit** ile yüklenir.
* **BÖLÜM B (12–16):** kilit doğrulanır, test referansları okunur, metrikler, zamansal analiz, hata analizi ve şekiller.

Bütün hücreler yukarıdan aşağıya ek elle düzenleme gerektirmeden çalışır. Veri seti yoksa resmî adresten indirilir.

## 0. Kurulum ve ortam

In [ ]:
import os, sys, json, time, hashlib, zipfile, urllib.request, platform, copy, re, warnings
from pathlib import Path

import subprocess
for _pkg in ("tifffile", "imagecodecs"):          # GOWT1 TIFF'leri LZW sıkıştırmalı → imagecodecs gerekir
    try:
        __import__(_pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _pkg])
import tifffile
import imagecodecs

import numpy as np
import pandas as pd
import cv2
import scipy
import skimage
import matplotlib
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from skimage import exposure, measure, morphology, segmentation, feature

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

STUDENT_ID = "244329059"
SEED = int(STUDENT_ID[-5:]) + 2026          # 29059 + 2026 = 31085

ROOT = Path(os.environ.get("PROJECT_ROOT", ".")).resolve()        # sonuç dosyaları repo köküne yazılır
DATA_PARENT = Path(os.environ.get("GOWT1_PARENT", ROOT / "data"))
DATA_ROOT = DATA_PARENT / "Fluo-N2DH-GOWT1"
PLOTS = ROOT / "plots"                        # veri görüntüsü İÇERMEYEN grafikler (GitHub'a girer)
PRIV = ROOT / "figures_private"               # veri görüntüsü İÇEREN şekiller (yalnızca rapor; .gitignore)
for p in (PLOTS, PRIV):
    p.mkdir(parents=True, exist_ok=True)

DATA_URL = "https://data.celltrackingchallenge.net/training-datasets/Fluo-N2DH-GOWT1.zip"

env = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
       "opencv": cv2.__version__, "scipy": scipy.__version__, "scikit-image": skimage.__version__,
       "matplotlib": matplotlib.__version__, "tifffile": tifffile.__version__, "imagecodecs": imagecodecs.__version__,
       "platform": platform.platform(), "seed": SEED}
(ROOT / "environment.json").write_text(json.dumps(env, indent=2))
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 150, "font.size": 9,
                     "axes.spines.top": False, "axes.spines.right": False})
def tl():
    plt.tight_layout(rect=(0, 0, 1, 0.94) if plt.gcf()._suptitle is not None else (0, 0, 1, 1))


COL = {"gt": "#222222", "base": "#E07B39", "final": "#2F6DB5", "c": "#4BA36B", "grey": "#9A9A9A"}
print(json.dumps(env, indent=2))

## 1. Veri setinin indirilmesi

In [ ]:
if not (DATA_ROOT / "01").exists():
    DATA_PARENT.mkdir(parents=True, exist_ok=True)
    zpath = DATA_PARENT / "Fluo-N2DH-GOWT1.zip"
    if not zpath.exists():
        print("İndiriliyor:", DATA_URL)
        urllib.request.urlretrieve(DATA_URL, zpath)
    with zipfile.ZipFile(zpath) as z:
        z.extractall(DATA_PARENT)
assert (DATA_ROOT / "01").exists() and (DATA_ROOT / "02").exists(), f"Veri bulunamadı: {DATA_ROOT}"
print("Veri kökü:", DATA_ROOT)
print(sorted(p.name for p in DATA_ROOT.iterdir()))


def frames_of(seq):
    return sorted(int(re.findall(r"\d+", f.stem)[0]) for f in (DATA_ROOT / seq).glob("t*.tif"))


def read_raw(seq, t):
    return tifffile.imread(DATA_ROOT / seq / f"t{t:03d}.tif")


FRAMES = {s: frames_of(s) for s in ("01", "02")}
print({s: (len(v), v[0], v[-1]) for s, v in FRAMES.items()})

## 2. Ön inceleme: boyut, veri tipi, yoğunluk istatistikleri ve histogramlar

In [ ]:
rows = []
for s in ("01", "02"):
    for t in FRAMES[s]:
        im = read_raw(s, t)
        p = np.percentile(im, [1, 50, 99, 99.9])
        rows.append({"sequence": s, "frame": t, "shape": "x".join(map(str, im.shape)), "dtype": str(im.dtype),
                     "min": int(im.min()), "max": int(im.max()), "mean": float(im.mean()), "std": float(im.std()),
                     "p1": p[0], "p50": p[1], "p99": p[2], "p99_9": p[3],
                     "fg_frac_otsu": float((im > skimage.filters.threshold_otsu(im)).mean())})
overview = pd.DataFrame(rows)
overview.to_csv(ROOT / "data_overview.csv", index=False)
summary = overview.groupby("sequence").agg(
    n=("frame", "size"), shape=("shape", "first"), dtype=("dtype", "first"),
    min=("min", "min"), max=("max", "max"), mean_mean=("mean", "mean"), mean_std=("mean", "std"),
    p50=("p50", "mean"), p99=("p99", "mean"), fg_frac=("fg_frac_otsu", "mean"))
summary.to_csv(ROOT / "data_overview_summary.csv")
print(summary.round(3))

RAW_DTYPE = overview["dtype"].iloc[0]
# 8 bit değilse, yalnızca SEKANS 01 (geliştirme/doğrulama kaynağı) üzerinden sabit bir doğrusal ölçek belirlenir.
SCALE = None if RAW_DTYPE == "uint8" else float(overview.query("sequence=='01'")["p99_9"].max())
print("Ham veri tipi:", RAW_DTYPE, "| 8 bit ölçeği:", SCALE)

# Histogramlar (veri görüntüsü değil; GitHub'a girebilir)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), sharey=True)
for ax, s in zip(axes, ("01", "02")):
    fr = FRAMES[s]
    for t, c in zip([fr[0], fr[len(fr) // 2], fr[-1]], ["#2F6DB5", "#E07B39", "#4BA36B"]):
        im = read_raw(s, t)
        h, edges = np.histogram(im, bins=128, range=(0, np.iinfo(im.dtype).max if im.dtype.kind == "u" else im.max()))
        ax.semilogy(edges[:-1], h + 1, color=c, lw=1.2, label=f"t{t:03d}")
    ax.set_title(f"Sekans {s}: yoğunluk histogramı (log)")
    ax.set_xlabel("piksel yoğunluğu"); ax.legend(frameon=False)
axes[0].set_ylabel("piksel sayısı + 1")
tl(); plt.savefig(PLOTS / "data_histograms.png"); plt.show()

fig, ax = plt.subplots(figsize=(10, 2.8))
for s, c in (("01", "#2F6DB5"), ("02", "#E07B39")):
    d = overview.query("sequence==@s")
    ax.plot(d["frame"], d["mean"], color=c, label=f"Seq {s} ortalama")
    ax.plot(d["frame"], d["p99"], color=c, ls="--", label=f"Seq {s} p99")
ax.set_xlabel("kare"); ax.set_ylabel("yoğunluk"); ax.legend(frameon=False, ncol=4)
ax.set_title("Kareler boyunca parlaklık değişimi")
tl(); plt.savefig(PLOTS / "data_intensity_over_time.png"); plt.show()

# Örnek görüntüler (veri görüntüsü → yalnızca özel klasör)
fig, axes = plt.subplots(1, 4, figsize=(14, 3.8))
for ax, (s, t) in zip(axes, [("01", FRAMES["01"][0]), ("01", FRAMES["01"][-1]), ("02", FRAMES["02"][0]), ("02", FRAMES["02"][-1])]):
    im = read_raw(s, t).astype(float)
    ax.imshow(im, cmap="gray", vmin=np.percentile(im, 1), vmax=np.percentile(im, 99.8))
    ax.set_title(f"Seq {s} t{t:03d} (görüntüleme için p1–p99.8)"); ax.axis("off")
tl(); plt.savefig(PRIV / "data_examples.png"); plt.show()

## 3. Geliştirme, doğrulama ve test karelerinin seçimi (seed = 31085)
* Seq 01: önce 5 doğrulama, ardından kalan uygun karelerden 10 geliştirme karesi. İki grup arasında **en az 5 ara kare** (|Δt| ≥ 6).
* Seq 02: 3 farklı başlangıç noktası, her birinden 5 ardışık kare; bloklar arasında en az 5 ara kare.
* Koşul sağlanmazsa aynı `np.random.default_rng(SEED)` akışından yeniden çekiliş yapılır (deterministik tekrar); deneme sayısı kaydedilir.
* Seçim, hiçbir algoritma çalıştırılmadan önce yapılır ve sonradan değiştirilmez.

In [ ]:
N_DEV, N_VAL, N_BLOCKS, BLOCK_LEN, MIN_GAP, MAX_TRIES = 10, 5, 3, 5, 5, 10_000


def far_enough(t, others, gap=MIN_GAP):
    return all(abs(t - o) >= gap + 1 for o in others)


def select_frames(frames01, frames02, seed=SEED):
    rng = np.random.default_rng(seed)
    pool01 = np.asarray(sorted(frames01))
    for a1 in range(1, MAX_TRIES + 1):
        val = sorted(rng.choice(pool01, size=N_VAL, replace=False).tolist())
        allowed = [t for t in pool01 if far_enough(t, val)]
        if len(allowed) >= N_DEV:
            dev = sorted(rng.choice(allowed, size=N_DEV, replace=False).tolist())
            break
    else:
        raise RuntimeError("Seq 01 koşulları sağlanamadı")
    f02 = set(frames02)
    starts_pool = np.array([t for t in sorted(frames02) if all(t + k in f02 for k in range(BLOCK_LEN))])
    for a2 in range(1, MAX_TRIES + 1):
        st = sorted(rng.choice(starts_pool, size=N_BLOCKS, replace=False).tolist())
        if all(st[i + 1] >= st[i] + BLOCK_LEN + MIN_GAP for i in range(N_BLOCKS - 1)):
            break
    else:
        raise RuntimeError("Seq 02 koşulları sağlanamadı")
    rows = [("development", "01", t, "") for t in dev] + [("validation", "01", t, "") for t in val]
    rows += [("test", "02", s + k, f"B{b}") for b, s in enumerate(st, 1) for k in range(BLOCK_LEN)]
    df = pd.DataFrame(rows, columns=["group", "sequence", "frame", "test_block"])
    df["file"] = df["sequence"] + "/t" + df["frame"].map("{:03d}".format) + ".tif"
    df["seed"], df["attempts_seq01"], df["attempts_seq02"] = seed, a1, a2
    return df


SEL = select_frames(FRAMES["01"], FRAMES["02"])
SEL.to_csv(ROOT / "selected_frames.csv", index=False)

# Kural kontrolleri
_dev = SEL.query("group=='development'").frame.tolist(); _val = SEL.query("group=='validation'").frame.tolist()
assert len(_dev) == 10 and len(_val) == 5 and not set(_dev) & set(_val)
assert min(abs(d - v) for d in _dev for v in _val) >= MIN_GAP + 1
_st = SEL.query("group=='test'").groupby("test_block").frame.min().sort_values().tolist()
assert all(b - a >= BLOCK_LEN + MIN_GAP for a, b in zip(_st, _st[1:]))
print(SEL.groupby("group").frame.apply(list))
print("Dev–val en küçük zamansal uzaklık:", min(abs(d - v) for d in _dev for v in _val))

## 4. Referans hücre sayıları
Referans sayı = `0x_GT/TRA/man_trackTTT.tif` içindeki sıfırdan farklı **farklı etiket** sayısı.
**Test referanslarını okuyan fonksiyon, kilit dosyası oluşana kadar Seq 02 için hata verir** (sızıntı koruması).

In [ ]:
LOCK_CSV = ROOT / "test_predictions_locked.csv"
LOCK_HASH = ROOT / "test_lock.sha256"


def read_tra(seq, t):
    if seq == "02" and not LOCK_HASH.exists():
        raise PermissionError("Test referansları, test tahminleri kilitlenmeden okunamaz!")
    return tifffile.imread(DATA_ROOT / f"{seq}_GT" / "TRA" / f"man_track{t:03d}.tif")


def reference_count(seq, t):
    ids = np.unique(read_tra(seq, t))
    return int((ids > 0).sum())


DEV = SEL.query("group=='development'").reset_index(drop=True).copy()
VAL = SEL.query("group=='validation'").reset_index(drop=True).copy()
TEST = SEL.query("group=='test'").reset_index(drop=True).copy()
for d in (DEV, VAL):
    d["ref_count"] = [reference_count(s, t) for s, t in zip(d.sequence, d.frame)]
    d["tra_exists"] = True
pd.concat([DEV, VAL]).to_csv(ROOT / "references_dev_val.csv", index=False)
# Test karelerinde yalnızca TRA dosyasının VARLIĞI kontrol edilir (içerik okunmaz)
TEST["tra_exists"] = [(DATA_ROOT / "02_GT" / "TRA" / f"man_track{t:03d}.tif").exists() for t in TEST.frame]
print(pd.concat([DEV, VAL])[["group", "frame", "ref_count"]].to_string(index=False))
print("Test TRA dosyaları mevcut mu:", TEST.tra_exists.all())

# Görsel kontrol: geliştirme görüntüleri + TRA işaretçileri
fig, axes = plt.subplots(2, 5, figsize=(16, 6.8))
for ax, (_, r) in zip(axes.ravel(), DEV.iterrows()):
    im = read_raw(r.sequence, r.frame).astype(float)
    ax.imshow(im, cmap="gray", vmin=np.percentile(im, 1), vmax=np.percentile(im, 99.8))
    tra = read_tra(r.sequence, r.frame)
    for reg in measure.regionprops(tra):
        y, x = reg.centroid
        ax.plot(x, y, "o", ms=4, mfc="none", mec="#FF3B3B", mew=1)
    ax.set_title(f"dev t{r.frame:03d}: referans = {r.ref_count}"); ax.axis("off")
tl(); plt.savefig(PRIV / "dev_reference_check.png"); plt.show()

## 5. Pipeline kütüphanesi
Her pipeline bir konfigürasyon sözlüğüdür: `intensity → filters → segmentation → morphology → split → min_area → sayım`.
Kullanılan hazır fonksiyonlar ve işlevleri:
* `cv2.equalizeHist` (global histogram eşitleme), `cv2.createCLAHE` (blok bazlı, kırpma limitli eşitleme),
  `skimage.exposure.match_histograms` (referans dağılımına eşleme), yüzdelik tabanlı doğrusal germe.
* `cv2.GaussianBlur`, `cv2.medianBlur`, `cv2.filter2D` (3×3 Laplasyen keskinleştirme çekirdeği), unsharp masking.
* `cv2.threshold(..., THRESH_OTSU)` (sınıf içi varyansı en aza indiren global eşik), `cv2.adaptiveThreshold`
  (Gauss ağırlıklı yerel ortalama − C), `cv2.kmeans` (yoğunluk kümeleme), sabit eşik.
* `cv2.erode/dilate/morphologyEx` (elips yapılandırma elemanı), `ndi.binary_fill_holes`.
* `ndi.distance_transform_edt` + `feature.peak_local_max` + `segmentation.watershed` (temas eden hücreleri ayırma).

In [ ]:
MATCH_REF = {"img": None, "frame": None}


def to_uint8(img):
    if img.dtype == np.uint8:
        return img
    return np.clip(img.astype(np.float32) / SCALE * 255.0, 0, 255).astype(np.uint8)


def apply_intensity(u8, p):
    m = p.get("method", "none")
    if m == "none":
        return u8
    if m == "stretch":
        lo, hi = np.percentile(u8, (p.get("p_low", 1), p.get("p_high", 99.8)))
        f = (u8.astype(np.float32) - lo) / max(hi - lo, 1) * 255
        return np.clip(f, 0, 255).astype(np.uint8)
    if m == "histeq":
        return cv2.equalizeHist(u8)
    if m == "clahe":
        t = p.get("tile", 8)
        return cv2.createCLAHE(clipLimit=p.get("clip", 2.0), tileGridSize=(t, t)).apply(u8)
    if m == "match":
        return np.clip(exposure.match_histograms(u8, MATCH_REF["img"]), 0, 255).astype(np.uint8)
    raise ValueError(m)


SHARPEN_KERNEL = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], np.float32)


def apply_filter(u8, p):
    m = p["method"]
    if m == "gauss":
        return cv2.GaussianBlur(u8, (p["k"], p["k"]), 0)
    if m == "median":
        return cv2.medianBlur(u8, p["k"])
    if m == "sharpen":
        return np.clip(cv2.filter2D(u8.astype(np.float32), -1, SHARPEN_KERNEL), 0, 255).astype(np.uint8)
    if m == "unsharp":
        f = u8.astype(np.float32)
        blur = cv2.GaussianBlur(f, (p.get("k", 9), p.get("k", 9)), 0)
        return np.clip(f + p.get("amount", 1.0) * (f - blur), 0, 255).astype(np.uint8)
    raise ValueError(m)


def segment(u8, p):
    m = p["method"]
    if m == "fixed":
        return u8 > p["t"]
    if m == "otsu":
        _, bw = cv2.threshold(u8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        return bw > 0
    if m == "adaptive":
        bw = cv2.adaptiveThreshold(u8, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, p["block"], p["C"])
        return bw > 0
    if m == "kmeans":
        z = u8.reshape(-1, 1).astype(np.float32)
        cv2.setRNGSeed(SEED)
        crit = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.5)
        _, lab, cen = cv2.kmeans(z, p["k"], None, crit, 2, cv2.KMEANS_PP_CENTERS)
        order = np.argsort(cen.ravel())
        return np.isin(lab.ravel(), order[p.get("bg_clusters", 1):]).reshape(u8.shape)
    raise ValueError(m)


def _se(k):
    return cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))


def apply_morph(bw, ops):
    b = bw.astype(np.uint8)
    for o in ops:
        if o["op"] == "fill":
            b = ndi.binary_fill_holes(b).astype(np.uint8); continue
        cvop = {"erode": None, "dilate": None, "open": cv2.MORPH_OPEN, "close": cv2.MORPH_CLOSE}[o["op"]]
        if o["op"] == "erode":
            b = cv2.erode(b, _se(o["k"]))
        elif o["op"] == "dilate":
            b = cv2.dilate(b, _se(o["k"]))
        else:
            b = cv2.morphologyEx(b, cvop, _se(o["k"]))
    return b.astype(bool)


def split_and_label(bw, sp, min_area):
    if min_area:                                      # alanı min_area'dan küçük bileşenleri sil (sürümden bağımsız)
        cc0 = measure.label(bw, connectivity=2)
        areas0 = np.bincount(cc0.ravel())
        bw = bw & (areas0[cc0] >= min_area)
    if sp.get("method", "none") == "none":
        return measure.label(bw, connectivity=2), None
    dist = ndi.distance_transform_edt(bw)
    if sp.get("sigma", 0):
        dist = ndi.gaussian_filter(dist, sp["sigma"])
    cc = measure.label(bw, connectivity=2)
    peaks = feature.peak_local_max(dist, min_distance=sp["min_distance"], threshold_abs=sp.get("min_peak", 3),
                                   labels=cc, exclude_border=False)
    markers = np.zeros(bw.shape, np.int32)
    markers[tuple(peaks.T)] = np.arange(1, len(peaks) + 1)
    lab = segmentation.watershed(-dist, markers, mask=bw)
    rest = (cc > 0) & (lab == 0)                       # tepe bulunamayan küçük bileşenler kaybolmasın
    if rest.any():
        extra = measure.label(rest, connectivity=2)
        lab = np.where(extra > 0, extra + lab.max(), lab)
    if min_area:                                      # bölünme kırıntılarını at
        areas = np.bincount(lab.ravel())
        lab[np.isin(lab, np.where(areas < min_area)[0][1:])] = 0
        lab = segmentation.relabel_sequential(lab)[0]
    return lab, dist


def run_pipeline(raw, cfg, keep=False):
    S = {}
    u8 = to_uint8(raw)
    x = apply_intensity(u8, cfg["intensity"])
    if keep: S["input"], S["intensity"] = u8, x
    for f in cfg["filters"]:
        x = apply_filter(x, f)
    if keep: S["filtered"] = x
    bw = segment(x, cfg["segmentation"])
    if keep: S["binary"] = bw
    bw = apply_morph(bw, cfg["morphology"])
    if keep: S["morph"] = bw
    lab, dist = split_and_label(bw, cfg["split"], cfg["min_area"])
    if keep: S["labels"], S["distance"] = lab, dist
    return int(lab.max()), lab, S


def make_cfg(intensity=None, filters=None, seg=None, morph=None, split=None, min_area=0):
    return {"intensity": intensity or {"method": "none"}, "filters": filters or [],
            "segmentation": seg or {"method": "otsu"}, "morphology": morph or [],
            "split": split or {"method": "none"}, "min_area": int(min_area)}


def _kv(d, skip=("method",)):
    s = ",".join(f"{k}={v}" for k, v in d.items() if k not in skip)
    return f"({s})" if s else ""


def describe(cfg):
    fl = " → ".join(f["method"] + (f"{f['k']}×{f['k']}" if f["method"] in ("gauss", "median") else _kv(f)) for f in cfg["filters"]) or "yok"
    mo = " → ".join(o["op"] + (str(o["k"]) if "k" in o else "") for o in cfg["morphology"]) or "yok"
    sp = cfg["split"]
    return {"intensity": cfg["intensity"]["method"] + _kv(cfg["intensity"]), "filter": fl,
            "segmentation": cfg["segmentation"]["method"] + _kv(cfg["segmentation"]), "morphology": mo,
            "split": "yok" if sp["method"] == "none" else f"watershed(md={sp['min_distance']},σ={sp.get('sigma', 0)})",
            "min_area": cfg["min_area"]}

## 6. Değerlendirme yardımcıları
Süre: görüntü bellekte iken `run_pipeline` çağrısının duvar saati süresi (disk okuma hariç).

In [ ]:
RAW = {}


def raw_img(seq, t):
    if (seq, t) not in RAW:
        RAW[(seq, t)] = read_raw(seq, t)
    return RAW[(seq, t)]


def metrics(y, yhat):
    y, yhat = np.asarray(y, float), np.asarray(yhat, float)
    e = yhat - y
    return {"MAE": np.mean(np.abs(e)), "MAE_std": np.std(np.abs(e), ddof=1) if len(e) > 1 else 0.0,
            "RMSE": np.sqrt(np.mean(e ** 2)), "Bias": np.mean(e), "ExactAcc": np.mean(e == 0)}


_CACHE = {}


def evaluate(cfg, df):
    key = (json.dumps(cfg, sort_keys=True), tuple(zip(df.sequence, df.frame)))
    if key in _CACHE:
        return _CACHE[key]
    preds, times = [], []
    for s, t in zip(df.sequence, df.frame):
        im = raw_img(s, t)
        t0 = time.perf_counter()
        c, _, _ = run_pipeline(im, cfg)
        times.append(time.perf_counter() - t0)
        preds.append(c)
    m = metrics(df.ref_count, preds)
    m["time_ms"] = 1000 * float(np.mean(times))
    _CACHE[key] = (m, preds)
    return m, preds


def error_breakdown(lab, tra):
    gt_ids = [i for i in np.unique(tra) if i > 0]
    hit, missed, split = {}, 0, 0
    for g in gt_ids:
        under = lab[tra == g]
        labs, cnt = np.unique(under[under > 0], return_counts=True)
        if len(labs) == 0:
            missed += 1; continue
        if len(labs) >= 2: split += 1
        hit.setdefault(int(labs[np.argmax(cnt)]), []).append(g)
    merged = sum(len(v) - 1 for v in hit.values() if len(v) > 1)
    fp = int(lab.max()) - len(hit)
    return {"ref": len(gt_ids), "pred": int(lab.max()), "missed": missed, "merged_extra": merged,
            "split_markers": split, "false_pos_labels": fp}


def error_table(cfg, df, tag):
    rows = []
    for s, t in zip(df.sequence, df.frame):
        c, lab, _ = run_pipeline(raw_img(s, t), cfg)
        rows.append({"set": tag, "frame": t, **error_breakdown(lab, read_tra(s, t))})
    return pd.DataFrame(rows)


EXPS, SWEEPS = [], []


def run_exp(group, changed, cfg, note=""):
    """Bir deneysel konfigürasyonu geliştirme kümesinde değerlendirir ve kaydeder (aynı konfigürasyon tekrar sayılmaz)."""
    js = json.dumps(cfg, sort_keys=True)
    for e in EXPS:
        if e["config_json"] == js:
            if changed not in e["note"] and changed != e["changed"]:
                e["note"] = (e["note"] + "; " if e["note"] else "") + f"= {changed}"
                print(f"{e['exp_id']} tekrar: '{changed}' ile aynı konfigürasyon (yeniden sayılmadı)")
            return e
    m, preds = evaluate(cfg, DEV)
    e = {"exp_id": f"E{len(EXPS) + 1:02d}", "group": group, "changed": changed, **describe(cfg),
         "dev_MAE": m["MAE"], "dev_MAE_std": m["MAE_std"], "dev_RMSE": m["RMSE"], "dev_Bias": m["Bias"],
         "dev_ExactAcc": m["ExactAcc"], "time_ms": m["time_ms"], "dev_preds": " ".join(map(str, preds)),
         "note": note, "config_json": js}
    EXPS.append(e)
    print(f"{e['exp_id']} [{group}] {changed:<38s} MAE={m['MAE']:.2f}±{m['MAE_std']:.2f} RMSE={m['RMSE']:.2f} "
          f"Bias={m['Bias']:+.2f} t={m['time_ms']:.0f}ms")
    return e


def sweep(name, param, values, cfg_fn, df=None):
    """Parametre taraması (yalnızca geliştirme kümesi). Sonuç SWEEPS'e yazılır; en iyi değer döner (eşitlikte ilk/küçük)."""
    df = DEV if df is None else df
    res = []
    for v in values:
        m, _ = evaluate(cfg_fn(v), df)
        res.append((v, m["MAE"], m["RMSE"], m["Bias"]))
        SWEEPS.append({"sweep": name, "param": param, "value": str(v), "dev_MAE": m["MAE"], "dev_RMSE": m["RMSE"],
                       "dev_Bias": m["Bias"]})
    best = min(res, key=lambda r: (round(r[1], 6), round(r[2], 6)))
    print(f"[sweep] {name}: en iyi {param}={best[0]} (MAE={best[1]:.2f})")
    return best[0]


def best_of(exps):
    """En düşük geliştirme MAE; eşitlikte RMSE, sonra süre."""
    return min(exps, key=lambda e: (round(e["dev_MAE"], 6), round(e["dev_RMSE"], 6), e["time_ms"]))


def cfg_of(e):
    return json.loads(e["config_json"])

## 7. Baseline: gri seviye → Otsu → bağlı bileşen etiketleme → sayım
Yalnızca küçük gürültü lekelerini dışarıda bırakmak için sabit bir minimum alan eşiği geliştirme kümesinde seçilir.

In [ ]:
AREA_GRID = [0, 25, 50, 100, 150, 200, 300, 400, 600, 800, 1000]
base_area = sweep("baseline_min_area", "min_area", AREA_GRID, lambda a: make_cfg(min_area=a))
BASELINE = make_cfg(min_area=base_area)
E_BASE = run_exp("baseline", f"Otsu + CCL, min_area={base_area}", BASELINE)

# Baseline görselleştirme (ilk geliştirme karesi)
r0 = DEV.iloc[0]
c, lab, S = run_pipeline(raw_img(r0.sequence, r0.frame), BASELINE, keep=True)
fig, ax = plt.subplots(1, 3, figsize=(14, 4.8))
ax[0].imshow(S["input"], cmap="gray"); ax[0].set_title(f"Girdi (t{r0.frame:03d})")
ax[1].imshow(S["binary"], cmap="gray"); ax[1].set_title("Otsu ikili maske")
ax[2].imshow(skimage.color.label2rgb(lab, bg_label=0)); ax[2].set_title(f"CCL: {c} nesne (referans {r0.ref_count})")
for a in ax: a.axis("off")
tl(); plt.savefig(PRIV / "baseline_example.png"); plt.show()

## 8. Kontrollü deneyler (yalnızca geliştirme kümesi)
Aşamalı (açgözlü) tasarım: her aşamada **yalnızca o aşamanın yöntemi/parametresi değiştirilir**, diğer aşamalar sabit tutulur.
Aşamanın en iyi seçeneği (en düşük dev MAE; eşitlikte RMSE, sonra süre) bir sonraki aşamanın sabiti olur.

### 8.1 Yoğunluk dönüşümleri
Sabit bağlam: Gaussian 5×5 → Otsu → delik doldurma → min_area (baseline'dan).
Histogram eşleme referansı: **geliştirme kümesinden**, ortalama parlaklığı medyan olan kare.

In [ ]:
_dev_means = [float(raw_img(s, t).mean()) for s, t in zip(DEV.sequence, DEV.frame)]
_ref_idx = int(np.argsort(_dev_means)[len(_dev_means) // 2])
MATCH_REF["frame"] = int(DEV.frame.iloc[_ref_idx])
MATCH_REF["img"] = to_uint8(raw_img("01", MATCH_REF["frame"]))
print("Histogram eşleme referansı: Seq 01 t%03d" % MATCH_REF["frame"])

CTX_I = dict(filters=[{"method": "gauss", "k": 5}], seg={"method": "otsu"}, morph=[{"op": "fill"}], min_area=base_area)
clahe_clip = sweep("clahe_clip", "clip", [1.0, 2.0, 3.0, 4.0],
                   lambda v: make_cfg(intensity={"method": "clahe", "clip": v, "tile": 8}, **CTX_I))
clahe_tile = sweep("clahe_tile", "tile", [4, 8, 16],
                   lambda v: make_cfg(intensity={"method": "clahe", "clip": clahe_clip, "tile": v}, **CTX_I))
I_OPTS = [("orijinal (dönüşüm yok)", {"method": "none"}),
          ("contrast stretching p1–p99.8", {"method": "stretch", "p_low": 1, "p_high": 99.8}),
          ("histogram eşitleme", {"method": "histeq"}),
          (f"CLAHE clip={clahe_clip}, tile={clahe_tile}", {"method": "clahe", "clip": clahe_clip, "tile": clahe_tile}),
          (f"histogram eşleme (ref t{MATCH_REF['frame']:03d})", {"method": "match"})]
stage_I = [run_exp("yoğunluk", name, make_cfg(intensity=it, **CTX_I)) for name, it in I_OPTS]
I_BEST = cfg_of(best_of(stage_I))["intensity"]
print("→ Seçilen yoğunluk dönüşümü:", I_BEST)

# Şekil 2: yoğunluk dönüşümleri + histogramlar (aynı geliştirme karesi)
r2 = DEV.iloc[int(np.argmax(DEV.ref_count))]
u8 = to_uint8(raw_img(r2.sequence, r2.frame))
fig, axes = plt.subplots(2, 5, figsize=(17, 6.6), gridspec_kw={"height_ratios": [3, 1.3]})
for j, (name, it) in enumerate(I_OPTS):
    out = apply_intensity(u8, it)
    e = stage_I[j]
    axes[0, j].imshow(out, cmap="gray", vmin=0, vmax=255); axes[0, j].axis("off")
    axes[0, j].set_title(f"{name}\ndev MAE={e['dev_MAE']:.2f}", fontsize=8)
    axes[1, j].hist(out.ravel(), bins=128, range=(0, 255), color="#2F6DB5", log=True)
    axes[1, j].set_xlim(0, 255); axes[1, j].set_xlabel("yoğunluk")
axes[1, 0].set_ylabel("piksel (log)")
fig.suptitle(f"Şekil 2 — Yoğunluk dönüşümleri ve histogramları (Seq 01 t{r2.frame:03d}, referans {r2.ref_count} hücre)")
tl(); plt.savefig(PRIV / "fig2_intensity.png"); plt.show()

### 8.2 Yumuşatma ve keskinleştirme filtreleri
Sabit bağlam: seçilen yoğunluk dönüşümü → [filtre] → Otsu → delik doldurma → min_area.

In [ ]:
CTX_F = dict(intensity=I_BEST, seg={"method": "otsu"}, morph=[{"op": "fill"}], min_area=base_area)
F_OPTS = [("filtre yok", []),
          ("Gaussian 3×3", [{"method": "gauss", "k": 3}]),
          ("Gaussian 5×5", [{"method": "gauss", "k": 5}]),
          ("Gaussian 9×9", [{"method": "gauss", "k": 9}]),
          ("Median 5×5", [{"method": "median", "k": 5}]),
          ("Laplasyen keskinleştirme 3×3", [{"method": "sharpen"}]),
          ("Gaussian 5×5 → keskinleştirme", [{"method": "gauss", "k": 5}, {"method": "sharpen"}]),
          ("Gaussian 5×5 → unsharp (k=9, a=1)", [{"method": "gauss", "k": 5}, {"method": "unsharp", "k": 9, "amount": 1.0}])]
stage_F = [run_exp("filtre", name, make_cfg(filters=fl, **CTX_F)) for name, fl in F_OPTS]
F_BEST = cfg_of(best_of(stage_F))["filters"]
print("→ Seçilen filtre zinciri:", F_BEST)

### 8.3 Segmentasyon yöntemleri
Sabit bağlam: seçilen yoğunluk + filtre → [segmentasyon] → delik doldurma → min_area.
Sabit eşik değeri ve adaptif eşik parametreleri yalnızca geliştirme kümesinde taranır.

In [ ]:
CTX_S = dict(intensity=I_BEST, filters=F_BEST, morph=[{"op": "fill"}], min_area=base_area)
fixed_t = sweep("fixed_threshold", "T", list(range(4, 200, 4)), lambda v: make_cfg(seg={"method": "fixed", "t": v}, **CTX_S))
ad_grid = [(b, c) for b in (31, 51, 101, 201, 301) for c in (-2, -5, -10, -20, -30)]
ad_bc = sweep("adaptive_block_C", "(block,C)", ad_grid,
              lambda v: make_cfg(seg={"method": "adaptive", "block": v[0], "C": v[1]}, **CTX_S))
S_OPTS = [(f"sabit eşik T={fixed_t}", {"method": "fixed", "t": fixed_t}),
          ("Otsu", {"method": "otsu"}),
          (f"adaptif Gauss (block={ad_bc[0]}, C={ad_bc[1]})", {"method": "adaptive", "block": ad_bc[0], "C": ad_bc[1]}),
          ("K-means k=2", {"method": "kmeans", "k": 2, "bg_clusters": 1}),
          ("K-means k=3 (en koyu küme arka plan)", {"method": "kmeans", "k": 3, "bg_clusters": 1})]
stage_S = [run_exp("segmentasyon", name, make_cfg(seg=sg, **CTX_S)) for name, sg in S_OPTS]
E_S_BEST = best_of(stage_S)
S_BEST = cfg_of(E_S_BEST)["segmentation"]
print("→ Seçilen segmentasyon:", S_BEST)

### 8.4 Morfolojik işlemler, yapılandırma elemanı boyutu ve işlem sırası
Sabit bağlam: seçilen yoğunluk + filtre + segmentasyon → [morfoloji] → min_area. (Elips yapılandırma elemanı.)

In [ ]:
CTX_M = dict(intensity=I_BEST, filters=F_BEST, seg=S_BEST, min_area=base_area)
f = {"op": "fill"}
M_OPTS = [("morfoloji yok", []),
          ("delik doldurma", [f]),
          ("fill → erosion 5", [f, {"op": "erode", "k": 5}]),
          ("fill → dilation 5", [f, {"op": "dilate", "k": 5}]),
          ("fill → opening 3", [f, {"op": "open", "k": 3}]),
          ("fill → opening 7", [f, {"op": "open", "k": 7}]),
          ("fill → opening 11", [f, {"op": "open", "k": 11}]),
          ("fill → closing 3", [f, {"op": "close", "k": 3}]),
          ("fill → closing 7", [f, {"op": "close", "k": 7}]),
          ("fill → opening 7 → closing 7", [f, {"op": "open", "k": 7}, {"op": "close", "k": 7}]),
          ("fill → closing 7 → opening 7", [f, {"op": "close", "k": 7}, {"op": "open", "k": 7}]),
          ("closing 7 → fill → opening 7", [{"op": "close", "k": 7}, f, {"op": "open", "k": 7}])]
stage_M = [run_exp("morfoloji", name, make_cfg(morph=mo, **CTX_M)) for name, mo in M_OPTS]
M_BEST = cfg_of(best_of(stage_M))["morphology"]
print("→ Seçilen morfoloji:", M_BEST)

### 8.5 Temas eden hücrelerin ayrılması: distance transform + watershed
`min_distance` (iki çekirdek merkezi arasındaki en küçük piksel uzaklığı) ve uzaklık haritasının Gauss yumuşatması σ taranır.

In [ ]:
CTX_W = dict(intensity=I_BEST, filters=F_BEST, seg=S_BEST, morph=M_BEST, min_area=base_area)
md = sweep("watershed_min_distance", "min_distance", [5, 9, 13, 17, 21, 27, 35, 45, 60],
           lambda v: make_cfg(split={"method": "watershed", "min_distance": v, "sigma": 2}, **CTX_W))
sg_w = sweep("watershed_sigma", "sigma", [0, 2, 4],
             lambda v: make_cfg(split={"method": "watershed", "min_distance": md, "sigma": v}, **CTX_W))
W_OPTS = [("ayırma yok (CCL)", {"method": "none"}),
          (f"watershed md={md}, σ={sg_w}", {"method": "watershed", "min_distance": md, "sigma": sg_w}),
          (f"watershed md={max(5, md // 2)}, σ={sg_w} (aşırı bölme testi)", {"method": "watershed", "min_distance": max(5, md // 2), "sigma": sg_w}),
          (f"watershed md={md * 2}, σ={sg_w} (az bölme testi)", {"method": "watershed", "min_distance": md * 2, "sigma": sg_w})]
stage_W = [run_exp("ayırma", name, make_cfg(split=sp, **CTX_W)) for name, sp in W_OPTS]
W_BEST = {"method": "watershed", "min_distance": md, "sigma": sg_w}

In [ ]:
# Şekil 3: segmentasyon ve morfoloji çıktılarının aynı kırpıntı üzerinde karşılaştırılması
tra2 = read_tra(r2.sequence, r2.frame)
cent = np.array([reg.centroid for reg in measure.regionprops(tra2)])
W = 384
best_xy, best_n = (0, 0), -1
for y0 in range(0, u8.shape[0] - W + 1, 32):
    for x0 in range(0, u8.shape[1] - W + 1, 32):
        n = ((cent[:, 0] >= y0) & (cent[:, 0] < y0 + W) & (cent[:, 1] >= x0) & (cent[:, 1] < x0 + W)).sum()
        if n > best_n:
            best_xy, best_n = (y0, x0), n
CY, CX = best_xy
CROP = (slice(CY, CY + W), slice(CX, CX + W))


def _show_case(ax, cfg, title):
    c, lab, S = run_pipeline(raw_img(r2.sequence, r2.frame), cfg, keep=True)
    ax.imshow(skimage.color.label2rgb(lab[CROP], bg_label=0, image=S["input"][CROP], alpha=0.35, image_alpha=1))
    cc = cent[(cent[:, 0] >= CY) & (cent[:, 0] < CY + W) & (cent[:, 1] >= CX) & (cent[:, 1] < CX + W)]
    ax.plot(cc[:, 1] - CX, cc[:, 0] - CY, "x", color="#FF3B3B", ms=5, mew=1.2)
    ax.set_title(f"{title}\nsayım={c} / ref={r2.ref_count}", fontsize=8); ax.axis("off")


seg_cases = [(name, make_cfg(seg=sg, **CTX_S)) for name, sg in S_OPTS]
mor_cases = [(M_OPTS[i][0], make_cfg(morph=M_OPTS[i][1], **CTX_M)) for i in (0, 4, 6, 8, 9)]
mor_cases.append((W_OPTS[1][0], make_cfg(split=W_OPTS[1][1], **CTX_W)))
fig, axes = plt.subplots(2, 6, figsize=(19, 8.6), gridspec_kw={"hspace": 0.32})
for ax, (n, c) in zip(axes[0], seg_cases): _show_case(ax, c, n)
axes[0, 5].imshow(u8[CROP], cmap="gray"); axes[0, 5].set_title("girdi (kırpıntı)", fontsize=8); axes[0, 5].axis("off")
for ax, (n, c) in zip(axes[1], mor_cases): _show_case(ax, c, n)
fig.suptitle(f"Şekil 3 — Segmentasyon (üst) ve morfoloji/ayırma (alt) karşılaştırması · Seq 01 t{r2.frame:03d}, "
             f"kırpıntı y={CY}, x={CX}, {W}×{W} · kırmızı × = TRA referans işaretçisi")
tl(); plt.savefig(PRIV / "fig3_seg_morph.png"); plt.show()

## 9. Aday tam pipeline'lar ve deney tablosu
* **Pipeline A** — seçilen yoğunluk + filtre + segmentasyon + morfoloji, **ayırma yok** (bağlı bileşen sayımı).
* **Pipeline B** — A + distance transform/watershed ile temas eden hücrelerin ayrılması.
* **Pipeline C** — farklı bir segmentasyon ailesi (seçilenden sonraki en iyi yöntem) + aynı morfoloji + watershed.

Her adayın min_area eşiği yeniden geliştirme kümesinde seçilir.

In [ ]:
def tune_area(cfg, name):
    a = sweep(f"{name}_min_area", "min_area", AREA_GRID, lambda v: {**copy.deepcopy(cfg), "min_area": v})
    return {**copy.deepcopy(cfg), "min_area": a}


alt = best_of([e for e in stage_S if cfg_of(e)["segmentation"]["method"] != S_BEST["method"]])
S_ALT = cfg_of(alt)["segmentation"]
CANDIDATES = {
    "Pipeline A": tune_area(make_cfg(I_BEST, F_BEST, S_BEST, M_BEST, {"method": "none"}), "A"),
    "Pipeline B": tune_area(make_cfg(I_BEST, F_BEST, S_BEST, M_BEST, W_BEST), "B"),
    "Pipeline C": tune_area(make_cfg(I_BEST, F_BEST, S_ALT, M_BEST, W_BEST), "C"),
}
CAND_EXP = {k: run_exp("aday pipeline", k, v) for k, v in CANDIDATES.items()}

exp_df = pd.DataFrame(EXPS)
exp_df.to_csv(ROOT / "experiments.csv", index=False)
pd.DataFrame(SWEEPS).to_csv(ROOT / "parameter_sweeps.csv", index=False)
print(f"Toplam {len(exp_df)} kontrollü deney.")
print(exp_df[["exp_id", "group", "changed", "dev_MAE", "dev_MAE_std", "dev_RMSE", "dev_Bias", "time_ms"]].round(2).to_string(index=False))

# Deney grafiği: aşama bazında dev MAE (± std)
groups = [g for g in exp_df.group.unique()]
fig, axes = plt.subplots(1, len(groups), figsize=(3.0 * len(groups), 4.2), sharey=True,
                         gridspec_kw={"width_ratios": [max(1, (exp_df.group == g).sum()) for g in groups]})
for ax, g in zip(np.atleast_1d(axes), groups):
    d = exp_df[exp_df.group == g]
    best = d.dev_MAE.min()
    ax.bar(d.exp_id, d.dev_MAE, yerr=d.dev_MAE_std, capsize=2,
           color=[COL["final"] if v == best else "#B8C7DC" for v in d.dev_MAE])
    ax.set_title(g); ax.tick_params(axis="x", rotation=90)
np.atleast_1d(axes)[0].set_ylabel("Geliştirme MAE (± std)")
tl(); plt.savefig(PLOTS / "experiments_dev_mae.png"); plt.show()

# Parametre taramaları
sw = pd.DataFrame(SWEEPS)
names = [n for n in sw.sweep.unique() if not n.endswith("_min_area") or n == "baseline_min_area"]
fig, axes = plt.subplots(2, (len(names) + 1) // 2, figsize=(4 * ((len(names) + 1) // 2), 6.4))
for ax, n in zip(axes.ravel(), names):
    d = sw[sw.sweep == n]
    xs = np.arange(len(d))
    ax.plot(xs, d.dev_MAE, "o-", color=COL["final"], ms=3, label="MAE")
    ax.plot(xs, d.dev_Bias, "s--", color=COL["base"], ms=3, label="Bias")
    step = max(1, len(d) // 8)
    ax.set_xticks(xs[::step]); ax.set_xticklabels(d.value.iloc[::step], rotation=45, fontsize=7)
    ax.axhline(0, color=COL["grey"], lw=0.6); ax.set_title(n, fontsize=9); ax.set_xlabel(d.param.iloc[0])
for ax in axes.ravel()[len(names):]: ax.axis("off")
axes.ravel()[0].legend(frameon=False)
tl(); plt.savefig(PLOTS / "parameter_sweeps.png"); plt.show()

## 10. Doğrulama ve nihai pipeline seçimi
Kural: en düşük doğrulama MAE → eşitlikte RMSE → ortalama süre → pipeline karmaşıklığı (adım sayısı).

In [ ]:
def complexity(cfg):
    return (cfg["intensity"]["method"] != "none") + len(cfg["filters"]) + 1 + len(cfg["morphology"]) + \
           (cfg["split"]["method"] != "none") + (cfg["min_area"] > 0)


vrows = []
for name, cfg in {**CANDIDATES, "Baseline (referans)": BASELINE}.items():
    md_, _ = evaluate(cfg, DEV)
    mv, pv = evaluate(cfg, VAL)
    vrows.append({"pipeline": name, **describe(cfg), "dev_MAE": md_["MAE"], "val_MAE": mv["MAE"],
                  "val_MAE_std": mv["MAE_std"], "val_RMSE": mv["RMSE"], "val_Bias": mv["Bias"],
                  "val_ExactAcc": mv["ExactAcc"], "val_time_ms": mv["time_ms"], "complexity": complexity(cfg),
                  "val_preds": " ".join(map(str, pv)), "val_refs": " ".join(map(str, VAL.ref_count))})
vdf = pd.DataFrame(vrows)
cand = vdf[vdf.pipeline != "Baseline (referans)"].copy()
cand["_k"] = list(zip(cand.val_MAE.round(6), cand.val_RMSE.round(6), cand.val_time_ms, cand.complexity))
FINAL_NAME = cand.sort_values("_k").pipeline.iloc[0]
vdf["selection"] = np.where(vdf.pipeline == FINAL_NAME, "SEÇİLDİ", "")
vdf.drop(columns=[]).to_csv(ROOT / "validation_results.csv", index=False)
FINAL = CANDIDATES[FINAL_NAME]
print(vdf[["pipeline", "dev_MAE", "val_MAE", "val_MAE_std", "val_RMSE", "val_Bias", "val_time_ms", "selection"]].round(3).to_string(index=False))
print("\nNİHAİ PIPELINE:", FINAL_NAME, json.dumps(FINAL, ensure_ascii=False))

fig, ax = plt.subplots(figsize=(7, 3.4))
x = np.arange(len(vdf)); w = 0.38
ax.bar(x - w / 2, vdf.dev_MAE, w, label="Geliştirme MAE", color="#B8C7DC")
ax.bar(x + w / 2, vdf.val_MAE, w, yerr=vdf.val_MAE_std, capsize=3, label="Doğrulama MAE (± std)", color=COL["final"])
ax.set_xticks(x); ax.set_xticklabels(vdf.pipeline); ax.set_ylabel("MAE (hücre)"); ax.legend(frameon=False)
tl(); plt.savefig(PLOTS / "validation_mae.png"); plt.show()

### 10.1 Doğrulama hata analizi (TRA işaretçileriyle)
Her referans işaretçisinin hangi tahmin etiketine düştüğüne bakılarak hatalar ayrıştırılır:
**kaçırılan** (işaretçi arka planda), **birleşme** (bir etikette ≥2 işaretçi), **fazla bölme** (bir işaretçi ≥2 etikete yayılmış),
**yanlış pozitif** (hiç işaretçi içermeyen etiket).

In [ ]:
err_val = pd.concat([error_table(FINAL, VAL, "val_final"), error_table(BASELINE, VAL, "val_baseline")])
err_val.to_csv(ROOT / "error_analysis_validation.csv", index=False)
print(err_val.groupby("set")[["ref", "pred", "missed", "merged_extra", "split_markers", "false_pos_labels"]].sum())

## 11. Nihai pipeline'ın kilitlenmesi ve test tahminleri (referanslar OKUNMADAN)

In [ ]:
assert not LOCK_HASH.exists(), "Kilit zaten oluşturulmuş: test tahminleri yeniden üretilemez."
final_json = {"final_pipeline_name": FINAL_NAME, "final_pipeline": FINAL, "baseline": BASELINE,
              "candidates": CANDIDATES, "histogram_match_reference": {"sequence": "01", "frame": MATCH_REF["frame"]},
              "uint8_scale": SCALE, "seed": SEED, "student_id_last5": STUDENT_ID[-5:],
              "selection_rule": "min val MAE → RMSE → time → complexity",
              "locked_at_utc": pd.Timestamp.now("UTC").isoformat(), "environment": env}
(ROOT / "final_pipeline.json").write_text(json.dumps(final_json, indent=2, ensure_ascii=False))

lrows = []
for _, r in TEST.iterrows():
    im = raw_img(r.sequence, r.frame)
    t0 = time.perf_counter(); cb, _, _ = run_pipeline(im, BASELINE); tb = time.perf_counter() - t0
    t0 = time.perf_counter(); cf, _, _ = run_pipeline(im, FINAL); tf = time.perf_counter() - t0
    lrows.append({"test_block": r.test_block, "sequence": r.sequence, "frame": r.frame,
                  "baseline_pred": cb, "final_pred": cf, "baseline_time_ms": 1000 * tb, "final_time_ms": 1000 * tf})
pd.DataFrame(lrows).to_csv(LOCK_CSV, index=False)
digest = hashlib.sha256(LOCK_CSV.read_bytes() + (ROOT / "final_pipeline.json").read_bytes()).hexdigest()
LOCK_HASH.write_text(digest + "\n")
print(pd.DataFrame(lrows).to_string(index=False))
print("\nKİLİT SHA-256:", digest)
print(">>> Şimdi final_pipeline.json, test_predictions_locked.csv ve test_lock.sha256 dosyalarını AYRI BİR COMMIT ile GitHub'a yükleyin.")

# BÖLÜM B — Bağımsız test değerlendirmesi
Bu bölüm yalnızca kilit commit'i atıldıktan sonra çalıştırılır. Önce kilidin bozulmadığı doğrulanır.

In [ ]:
# Oturum yeniden başlatıldıysa: 0–6. bölümleri çalıştırıp doğrudan buraya gelinebilir; durum dosyalardan yüklenir.
if "FINAL" not in globals():
    _fj = json.loads((ROOT / "final_pipeline.json").read_text())
    FINAL_NAME, FINAL, BASELINE = _fj["final_pipeline_name"], _fj["final_pipeline"], _fj["baseline"]
    MATCH_REF["frame"] = _fj["histogram_match_reference"]["frame"]
    MATCH_REF["img"] = to_uint8(raw_img("01", MATCH_REF["frame"]))
    vdf = pd.read_csv(ROOT / "validation_results.csv")
    print("Durum dosyalardan yüklendi:", FINAL_NAME)

digest_now = hashlib.sha256(LOCK_CSV.read_bytes() + (ROOT / "final_pipeline.json").read_bytes()).hexdigest()
assert digest_now == LOCK_HASH.read_text().strip(), "Kilitli dosyalar değişmiş!"
print("Kilit doğrulandı:", digest_now)
locked = pd.read_csv(LOCK_CSV)

## 12. Test performansı (Tablo 4 ve Tablo 5)

In [ ]:
locked["ref_count"] = [reference_count("02", int(t)) for t in locked.frame]
locked["baseline_err"] = locked.baseline_pred - locked.ref_count
locked["final_err"] = locked.final_pred - locked.ref_count
locked["baseline_abs_err"] = locked.baseline_err.abs()
locked["final_abs_err"] = locked.final_err.abs()
srows = []
for name, col, tcol in (("Baseline", "baseline_pred", "baseline_time_ms"), ("Nihai pipeline", "final_pred", "final_time_ms")):
    m = metrics(locked.ref_count, locked[col])
    srows.append({"method": name, **m, "time_ms": locked[tcol].mean()})
summ = pd.DataFrame(srows)
locked.to_csv(ROOT / "test_results.csv", index=False)
summ.to_csv(ROOT / "test_summary.csv", index=False)
print(locked[["test_block", "frame", "ref_count", "baseline_pred", "final_pred", "baseline_abs_err", "final_abs_err"]].to_string(index=False))
print(summ.round(3).to_string(index=False))

# Doğrulama ↔ test karşılaştırması
vt = pd.DataFrame([
    {"method": "Baseline", "val_MAE": vdf.loc[vdf.pipeline == "Baseline (referans)", "val_MAE"].iloc[0],
     "test_MAE": summ.MAE.iloc[0], "val_RMSE": vdf.loc[vdf.pipeline == "Baseline (referans)", "val_RMSE"].iloc[0], "test_RMSE": summ.RMSE.iloc[0],
     "val_Bias": vdf.loc[vdf.pipeline == "Baseline (referans)", "val_Bias"].iloc[0], "test_Bias": summ.Bias.iloc[0]},
    {"method": f"Nihai ({FINAL_NAME})", "val_MAE": vdf.loc[vdf.pipeline == FINAL_NAME, "val_MAE"].iloc[0],
     "test_MAE": summ.MAE.iloc[1], "val_RMSE": vdf.loc[vdf.pipeline == FINAL_NAME, "val_RMSE"].iloc[0], "test_RMSE": summ.RMSE.iloc[1],
     "val_Bias": vdf.loc[vdf.pipeline == FINAL_NAME, "val_Bias"].iloc[0], "test_Bias": summ.Bias.iloc[1]}])
vt.to_csv(ROOT / "validation_vs_test.csv", index=False)
print(vt.round(3).to_string(index=False))

# Hücre sayısına göre işaretli hata (ölçek etkisi)
fig, ax = plt.subplots(figsize=(5.2, 3.6))
ax.scatter(locked.ref_count, locked.baseline_err, color=COL["base"], label="Baseline")
ax.scatter(locked.ref_count, locked.final_err, color=COL["final"], label="Nihai")
ax.axhline(0, color=COL["grey"], lw=0.8); ax.set_xlabel("referans hücre sayısı"); ax.set_ylabel("tahmin − referans")
ax.legend(frameon=False); tl(); plt.savefig(PLOTS / "test_error_vs_count.png"); plt.show()

## 13. Ardışık karelerde zamansal analiz (Şekil 5)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8), sharey=True)
trows = []
for ax, (b, d) in zip(axes, locked.groupby("test_block")):
    ax.plot(d.frame, d.ref_count, "o-", color=COL["gt"], lw=2, label="Referans (TRA)")
    ax.plot(d.frame, d.baseline_pred, "s--", color=COL["base"], label="Baseline")
    ax.plot(d.frame, d.final_pred, "^-", color=COL["final"], label=f"Nihai ({FINAL_NAME})")
    ax.set_title(f"Test bloğu {b}: Seq 02 t{d.frame.min():03d}–t{d.frame.max():03d}")
    ax.set_xlabel("kare"); ax.set_xticks(d.frame); ax.grid(alpha=0.3)
    dr, dbp, dfp = np.diff(d.ref_count), np.diff(d.baseline_pred), np.diff(d.final_pred)
    trows.append({"block": b, "frames": f"{d.frame.min()}–{d.frame.max()}",
                  "ref_range": f"{d.ref_count.min()}–{d.ref_count.max()}",
                  "ref_total_change": int(d.ref_count.iloc[-1] - d.ref_count.iloc[0]),
                  "baseline_total_change": int(d.baseline_pred.iloc[-1] - d.baseline_pred.iloc[0]),
                  "final_total_change": int(d.final_pred.iloc[-1] - d.final_pred.iloc[0]),
                  "baseline_step_MAE": float(np.mean(np.abs(dbp - dr))), "final_step_MAE": float(np.mean(np.abs(dfp - dr))),
                  "baseline_max_jump": int(np.max(np.abs(dbp))), "final_max_jump": int(np.max(np.abs(dfp))),
                  "ref_max_jump": int(np.max(np.abs(dr))),
                  "baseline_MAE": float(d.baseline_abs_err.mean()), "final_MAE": float(d.final_abs_err.mean()),
                  "final_Bias": float(d.final_err.mean())})
axes[0].set_ylabel("hücre sayısı"); axes[0].legend(frameon=False, fontsize=8)
fig.suptitle("Şekil 5 — Üç test bloğunda gerçek ve tahmin edilen hücre sayılarının zamansal değişimi")
tl(); plt.savefig(PLOTS / "fig5_temporal.png"); plt.show()
temporal = pd.DataFrame(trows); temporal.to_csv(ROOT / "temporal_analysis.csv", index=False)
print(temporal.round(2).to_string(index=False))

## 14. Test hata analizi ve başarılı / hatalı / zorlayıcı örnekler (Şekil 6)

In [ ]:
TEST_DF = locked.rename(columns={})[["sequence", "frame"]].copy()
TEST_DF["sequence"] = "02"
err_test = pd.concat([error_table(FINAL, TEST_DF, "test_final"), error_table(BASELINE, TEST_DF, "test_baseline")])
err_test.to_csv(ROOT / "error_analysis_test.csv", index=False)
print(err_test.groupby("set")[["ref", "pred", "missed", "merged_extra", "split_markers", "false_pos_labels"]].sum())

ef = err_test[err_test.set == "test_final"].merge(locked[["frame", "final_abs_err"]], on="frame")
ef["n_err"] = ef.missed + ef.merged_extra + ef.split_markers + ef.false_pos_labels
pick = {"Başarılı": ef.sort_values(["final_abs_err", "n_err"]).frame.iloc[0],
        "Hatalı": ef.sort_values(["final_abs_err", "n_err"], ascending=False).frame.iloc[0]}
rest = ef[~ef.frame.isin(pick.values())]
pick["Zorlayıcı (sayım tutsa da hata bileşenleri çok)"] = rest.sort_values(["n_err", "final_abs_err"], ascending=False).frame.iloc[0]

fig, axes = plt.subplots(1, 3, figsize=(17, 6))
for ax, (kind, t) in zip(axes, pick.items()):
    c, lab, S = run_pipeline(raw_img("02", int(t)), FINAL, keep=True)
    tra = read_tra("02", int(t))
    ax.imshow(S["input"], cmap="gray", vmax=np.percentile(S["input"], 99.8))
    ax.contour(lab > 0, levels=[0.5], colors=["#2FD3FF"], linewidths=0.6)
    ax.imshow(np.ma.masked_equal(segmentation.find_boundaries(lab, mode="inner"), 0), cmap="autumn", alpha=0.9)
    for reg in measure.regionprops(tra):
        ax.plot(reg.centroid[1], reg.centroid[0], "x", color="#FF3B3B", ms=5, mew=1.2)
    e = ef[ef.frame == t].iloc[0]
    ax.set_title(f"{kind}: Seq 02 t{int(t):03d}\nreferans={e.ref}, nihai={c} | kaçırılan={e.missed}, "
                 f"birleşme={e.merged_extra}, bölünme={e.split_markers}, YP={e.false_pos_labels}", fontsize=8)
    ax.axis("off")
fig.suptitle("Şekil 6 — Nihai pipeline: başarılı, hatalı ve zorlayıcı sayım örnekleri (kırmızı × = TRA referansı, renkli sınırlar = tahmin)")
tl(); plt.savefig(PRIV / "fig6_examples.png"); plt.show()

## 15. Şekil 1 (akış diyagramı) ve Şekil 4 (aşama aşama çıktılar)

In [ ]:
d = describe(FINAL)
steps = [("Girdi", "8 bit gri görüntü"), ("Yoğunluk dönüşümü", d["intensity"]), ("Filtreleme", d["filter"]),
         ("Segmentasyon", d["segmentation"]), ("Morfoloji", d["morphology"]),
         ("Küçük nesne eleme", f"alan < {FINAL['min_area']} px"), ("Nesne ayırma / etiketleme",
         d["split"] if d["split"] != "yok" else "bağlı bileşen (8-komşuluk)"), ("Çıktı", "hücre sayısı")]
import textwrap
fig, ax = plt.subplots(figsize=(13, 2.3)); ax.axis("off")
n = len(steps); bw = 1 / n
for i, (h, s_) in enumerate(steps):
    x0 = i * bw + 0.016
    ax.add_patch(matplotlib.patches.FancyBboxPatch((x0, 0.12), bw - 0.03, 0.74, boxstyle="round,pad=0.01",
                 fc="#EAF1FB" if 0 < i < n - 1 else "#D9D9D9", ec="#2F6DB5", lw=1.2, transform=ax.transAxes))
    xc = x0 + (bw - 0.03) / 2
    ax.text(xc, 0.72, textwrap.fill(h, 16), ha="center", va="center", fontsize=8.5, weight="bold", transform=ax.transAxes)
    ax.text(xc, 0.38, textwrap.fill(s_.replace(",", ", "), 18, break_long_words=False), ha="center", va="center", fontsize=7.5, transform=ax.transAxes)
    if i < n - 1:
        ax.annotate("", xy=(x0 + bw - 0.004, 0.49), xytext=(x0 + bw - 0.026, 0.49), xycoords="axes fraction",
                    arrowprops=dict(arrowstyle="->", color="#2F6DB5", lw=1.4))
ax.set_title(f"Şekil 1 — Nihai pipeline ({FINAL_NAME})", fontsize=10)
tl(); plt.savefig(PLOTS / "fig1_pipeline_flow.png"); plt.show()

r4 = DEV.iloc[int(np.argsort(DEV.ref_count.values)[len(DEV) // 2])]
c, lab, S = run_pipeline(raw_img(r4.sequence, r4.frame), FINAL, keep=True)
panels = [("1) Girdi", S["input"], "gray"), (f"2) Yoğunluk: {d['intensity']}", S["intensity"], "gray"),
          (f"3) Filtre: {d['filter']}", S["filtered"], "gray"), (f"4) Segmentasyon: {d['segmentation']}", S["binary"], "gray"),
          (f"5) Morfoloji: {d['morphology']}", S["morph"], "gray")]
if S["distance"] is not None:
    panels.append(("6) Uzaklık dönüşümü", S["distance"], "magma"))
panels.append((f"{len(panels) + 1}) Etiketler: {c} hücre (referans {r4.ref_count})",
               skimage.color.label2rgb(lab, bg_label=0, image=S["input"], alpha=0.4, image_alpha=1), None))
_nc = int(np.ceil(len(panels) / 2))
fig, axes = plt.subplots(2, _nc, figsize=(4.3 * _nc, 9.4), gridspec_kw={"hspace": 0.18})
for ax, (tt, im, cm) in zip(axes.ravel(), panels):
    ax.imshow(im, cmap=cm) if cm else ax.imshow(im); ax.set_title(tt, fontsize=8); ax.axis("off")
for ax in axes.ravel()[len(panels):]: ax.axis("off")
fig.suptitle(f"Şekil 4 — Nihai pipeline'ın aşama aşama çıktıları (geliştirme karesi Seq 01 t{r4.frame:03d})")
tl(); plt.savefig(PRIV / "fig4_stages.png"); plt.show()

## 16. Sonuç paketi
Rapor için bütün CSV/JSON ve şekiller `results_bundle.zip` içine toplanır (veri görüntüsü içeren şekiller dahil — bu zip GitHub'a **yüklenmez**).

In [ ]:
bundle = ROOT / "results_bundle.zip"
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(ROOT.glob("*.csv")) + sorted(ROOT.glob("*.json")) + [LOCK_HASH]:
        z.write(p, p.name)
    for p in sorted(PLOTS.glob("*.png")) + sorted(PRIV.glob("*.png")):
        z.write(p, f"{p.parent.name}/{p.name}")
print("Hazır:", bundle, f"{bundle.stat().st_size / 1e6:.1f} MB")
try:
    from google.colab import files  # Colab'da otomatik indir
    files.download(str(bundle))
except Exception:
    pass